In [1]:
import torch, torchaudio, librosa, torch.nn as nn, torch.nn.functional as F
from mpmath import diffun
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import math, glob, json
from pathlib import Path
from os.path import dirname


In [2]:
class CustomDataset(Dataset):
    def __init__(self, all_samples, window_size=512, debug = False, inference = False):
        self.all_samples = all_samples
        self.n_samples = len(all_samples)
        self.sample_rate = 22050
        self.debug = debug
        self.inference = inference
        self.spec_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80.0)

        self.window_size = window_size

    @classmethod
    def split(cls, n_samples=None, train_ratio=0.8, seed=42, **kwargs):
        """
        Creates non-overlapping train and test datasets.

        Args:
            n_samples:   max number of samples to use in total (None = use all)
            train_ratio: fraction of data to use for training (default 0.8)
            seed:        random seed for reproducibility
            **kwargs:    forwarded to __init__ (hop_length, window_size, etc.)
        """

        all_files = glob.glob(r"./processed/**/*_meta.pt", recursive=True)

        rng = np.random.default_rng(seed)
        if not kwargs['debug']:
            rng.shuffle(all_files)
        else: print("Using debug config")

        if n_samples is not None:
            all_files = all_files[:n_samples]

        split_idx = int(len(all_files) * train_ratio)
        train_files = all_files[:split_idx]
        test_files  = all_files[split_idx:]

        return cls(train_files, **kwargs), cls(test_files, **kwargs)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):

        data = torch.load(self.all_samples[idx])

        audio_file_name = data['AudioFilename']
        root_folder = data['BeatmapSetID']
        beatmap_id = data['BeatmapID']
        audio_path = fr"./processed/{root_folder}/{audio_file_name}"
        if self.debug:
            print(fr"{root_folder}/{beatmap_id}")

        spec_db_norm = torch.load(audio_path+'.pt')
        sequence=torch.load(fr"./processed/{root_folder}/{beatmap_id}_diff.pt")
        y, difficulty = sequence[:, :17], sequence[:, 17:]

        n_frames = len(y)
        if self.inference:
            return self.process_inference(y, difficulty, spec_db_norm, n_frames)
        else:
            return self.process_normal(y, difficulty, spec_db_norm, n_frames)


    def process_normal(self, y ,difficulty, spec_db_norm, n_frames):
        if n_frames > self.window_size:
            start_frame = torch.randint(0, n_frames - self.window_size, (1,)).item()
            X_window = spec_db_norm[:, start_frame:start_frame + self.window_size]
            y_window = y[start_frame:start_frame + self.window_size, :]

            difficulty_window = difficulty[start_frame:start_frame + self.window_size, :]
        else:
            pad_size = self.window_size - n_frames
            X_window = F.pad(spec_db_norm, (0, pad_size))
            y_window = F.pad(y, (0, 0, 0, pad_size))
            difficulty_window = F.pad(difficulty, (0, 0, 0, pad_size))
        return X_window, y_window, difficulty_window

    def process_inference(self, y ,difficulty, spec_db_norm, n_frames):
        if n_frames < self.window_size:
            pad_size = self.window_size - n_frames
            X_window = F.pad(spec_db_norm, (0, pad_size))
            y_window = F.pad(y, (0, 0, 0, pad_size))
            difficulty_window = F.pad(difficulty, (0, 0, 0, pad_size))
        else:
            raise Exception(f"Inference sequence must be less than window size! ({self.window_size})")
        return X_window, y_window, difficulty_window


In [3]:
class CNN(nn.Module):
    def __init__(self, out_size = 64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=out_size, kernel_size=3, padding=1)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, None)) #preserve time domain, compress only freq domain
        self.bn1 = nn.BatchNorm2d(num_features=32)
        self.bn2 = nn.BatchNorm2d(num_features=out_size)

    def forward(self,x):
        x=x.unsqueeze(1)
        x = self.bn1(F.relu(self.conv1(x)))
        x = self.bn2(F.relu(self.conv2(x)))
        x = self.avg_pool(x)
        return x

class AttentionBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        #Linear layers to project Q, K, V
        #Those are trainable parameters for a single head that we will tweak
        self.W_q = nn.Linear(input_dim, output_dim, bias=False)
        self.W_k = nn.Linear(input_dim, output_dim, bias=False)
        self.W_v = nn.Linear(input_dim, output_dim, bias=False)

    def forward(self, query, key, value):
        #project Q, K, V
        q_logits = self.W_q(query)
        k_logits = self.W_k(key)
        v_logits = self.W_v(value)

        attention, weights = self.scaled_dot_product_attention(q_logits, k_logits, v_logits)
        return attention, weights
        #We encapsulate the calculations explicitly which are done by each head. Q, K, V are projected independantly

    def scaled_dot_product_attention(self, q_logits, k_logits = None, v_logits = None):
        k_logits = k_logits if k_logits is not None else q_logits
        v_logits = v_logits if v_logits is not None else q_logits
        assert q_logits.size(-1) == k_logits.size(-1), "query and key must have the same embedding dimension"

        dim_k = q_logits.size(-1) #embed dimensions of key
        q_k = q_logits @ k_logits.transpose(-1, -2) / dim_k**0.5 # compute dot product to obtain similarity

        attn_weights = torch.softmax(q_k, dim=-1)

        #compute weighted sum of value vectors
        attention = attn_weights @ v_logits # attn = (bs, seq_len, embed_dim)
        return attention, attn_weights


class PositionalEmbedding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        #create a matrix that represents positional encoding for each token
        pe = torch.zeros(max_len, embed_dim)
        nominator = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # (max_len, 1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))

        pe[:, 0::2] = torch.sin(nominator * div_term)
        pe[:, 1::2] = torch.cos(nominator * div_term)

        pe=pe.unsqueeze(0)
        self.register_buffer('pe', pe, persistent=False)

    def forward(self,x):
        x = x + self.pe[:, :x.size(1), :]
        return x


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads" # required for consistensy so all heads have the same


        self.embed_per_head = embed_dim // num_heads
        self.heads = nn.ModuleList([AttentionBlock(input_dim=embed_dim, output_dim=self.embed_per_head) for _ in range(num_heads)])

        self.projection = nn.Linear(embed_dim, embed_dim, bias=False) #final projection of MHA



    def forward(self, query, key, value):
        attentions_list = []
        attention_weights_list = []

        for head in self.heads:
            attention, attention_weights = head(query, key, value) # for each head calculate its attention
            attentions_list.append(attention)
            attention_weights_list.append(attention_weights)

        #concatenate attention outputs and take avg of attn weights
        attentions, attention_weights = torch.cat(attentions_list, dim=2), torch.stack(attention_weights_list).mean(dim=0)
        return self.projection(attentions), attention_weights

class SoundSequenceAnalyzer(nn.Module):
    def __init__(self, time_window_size = 128, cnn_out = 64, freq_vec_len = 100, num_heads=1): #embed dim - vector length for a single frequency (n_bins)
        super().__init__()
        self.cnn_preprocessor = CNN(time_window_size=time_window_size, out_size = cnn_out)#extract local acoustic features

        self.input_projection = nn.Linear(in_features=cnn_out + 9, out_features=freq_vec_len, bias=False) #6 additional features for difficulty parameters

        self.positional_embedding = PositionalEmbedding(embed_dim=freq_vec_len, max_len=time_window_size)

        self.mha = MultiHeadAttention(embed_dim=freq_vec_len, num_heads=num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(freq_vec_len, freq_vec_len * 4),
            nn.ReLU(),
            nn.Linear(freq_vec_len * 4, freq_vec_len),
        )

        self.norm1 = nn.LayerNorm(freq_vec_len)
        self.norm2 = nn.LayerNorm(freq_vec_len)



        #predict different aspects
        self.hit_classifier = nn.Linear(freq_vec_len, 1) #Is there a hit object at this frame
        #self.pos = nn.Linear(freq_vec_len, 2) #X, Y
        self.pos = CoordMDNHead(freq_vec_len)
        self.obj_type = nn.Linear(freq_vec_len, 3) #circle, slider, spinner
        self.obj_attributes = nn.Linear(freq_vec_len, 3)# Predicts end_x, end_y, attr_val
        self.curve_classifier = nn.Linear(freq_vec_len, 4) # 4 classes: L, B, P, C
        self.anchor_count = nn.Linear(freq_vec_len, 4) # Predicts 0, 1, 2, or 3 anchors
        self.anchor_coords = nn.Linear(freq_vec_len, 6) # Predicts 3 pairs of (x,y)

    def forward(self, spectrogram, difficulty): #(B, 1, n_mels, T)

        x = self.cnn_preprocessor(spectrogram) #(B, 64, 1, Time) 64 feature maps, freq compressed to 1
        x = x.squeeze(2) #(B, 64, Time)
        x = x.permute(0, 2, 1) #(B, Time, 64) put time first, so each "token" is a 64-dim vector


        x = self.input_projection(torch.cat((x, difficulty), dim=-1)) #(B, Time, embed_dim). Concatenate with difficulty to distingush between different difficulties of the map
        x = self.positional_embedding(x) #(B, Time, embed_dim)

        #Perform self attention. Each frame looks at every other frame.
        attended, weights = self.mha(x, x, x)

        x = self.norm1(x + attended) #residual connection + layer norm
        x = self.norm2(x + self.ffn(x)) #(B, Time, embed_dim)

        is_object = self.hit_classifier(x)
        #coords = torch.sigmoid(self.pos(x)) #commented out for testing so that coords don't stuck at 0.5 0.5 (likely cause is sigmoid function). Trying to use simple linear output
        coords = self.pos(x)
        obj_type = self.obj_type(x)
        obj_attr = self.obj_attributes(x)
        curve_class = self.curve_classifier(x)
        anchor_count = self.anchor_count(x)
        anchor_coords = self.anchor_coords(x)
        return is_object, coords, obj_type, obj_attr, curve_class, anchor_count, anchor_coords

class CoordMDNHead(nn.Module):
    def __init__(self, in_features, n_components=5):
        super().__init__()
        self.n_components = n_components
        self.fc = nn.Linear(in_features, n_components + n_components*2 + n_components*2)

    def forward(self, x):
        out = self.fc(x)
        pi_logits = out[..., :self.n_components]
        mu = torch.sigmoid(out[..., self.n_components:self.n_components*3])
        log_sigma = out[..., self.n_components*3:]

        mu = mu.view(*mu.shape[:-1], self.n_components, 2)
        log_sigma = log_sigma.view(*log_sigma.shape[:-1], self.n_components, 2)
        return pi_logits, mu, log_sigma


In [4]:
def mdn_loss(pi_logits, mu, log_sigma, target):
    """
    pi_logits: [..., K]
    mu:        [..., K, 2]
    log_sigma: [..., K, 2]
    target:    [..., 2]
    """
    sigma = torch.exp(log_sigma).clamp(min=1e-4)
    target = target.unsqueeze(-2)

    log_p = -0.5 * (((target - mu)/ sigma) ** 2 + 2 * log_sigma + math.log(2 * math.pi))
    log_p = log_p.sum(dim=-1)

    log_pi = torch.log_softmax(pi_logits, dim=-1)
    log_mixture = torch.logsumexp(log_p + log_pi, dim=-1)

    return -log_mixture.mean()

In [5]:
def train(model, train_loader, test_loader ,device="cpu", epochs=200, lr=0.001, debug=False):
    print("Using device: ", device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    #
    is_object_lf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.0]).to(device))#missed true positive get 2x penalty
    obj_type_lf = nn.CrossEntropyLoss()
    attr_lf = nn.SmoothL1Loss()
    curve_lf = nn.CrossEntropyLoss()
    anchor_count_lf = nn.CrossEntropyLoss()
    anchor_coord_lf = nn.MSELoss(reduction='none')


    train_losses, val_losses = [], []
    best_loss = float('inf')
    epochs_wo_improvement = 0
    model = model.to(device)
    for epoch in range(epochs):
        model.train()
        if epochs_wo_improvement>9:
            print("Early stopping at epoch", epoch)
            break

        epoch_loss = 0.0
        for X, y, difficulty in train_loader:
            X, y, difficulty = X.to(device), y.to(device), difficulty.to(device)
            loss_coords = 0.0
            loss_obj_type = 0.0
            loss_attr = 0.0
            loss_curve = 0.0
            loss_anchor_count = 0.0
            loss_anchor_coords = 0.0
            is_object = y[:, :, 0].unsqueeze(-1)
            coords = y[:, :, 1:3]
            obj_type = y[:, :, 3:6]   # 3 classes: circle, slider, spinner
            obj_attr = y[:, :, 6:9]
            curve_targets = y[:, :, 9].long()
            is_obj_p, coords_p, type_p, attr_p, curve_p, anchor_count_p, anchor_coords_p = model(X, difficulty)
            loss_obj_all = is_object_lf(is_obj_p, is_object )

            mask = (is_object == 1).squeeze(-1)
            if mask.any():
                #loss_coords = coords_lf(coords_pred[mask], coords[mask])
                pi_logits, mu, log_sigma = coords_p
                loss_coords = mdn_loss(pi_logits[mask], mu[mask], log_sigma[mask], coords[mask])

                valid_type_preds = type_p[mask]
                valid_type_targets = obj_type[mask]
                loss_obj_type = obj_type_lf(valid_type_preds, valid_type_targets.argmax(dim=-1))
                loss_attr = attr_lf(attr_p[mask], obj_attr[mask])

                slider_mask = (obj_type[mask][:, 1] == 1.0)

                if slider_mask.any():
                    loss_curve = curve_lf(curve_p[mask][slider_mask], curve_targets[mask][slider_mask])
                    #Anchor Count Loss
                    target_anchor_counts = y[mask][slider_mask, 10].long()
                    loss_anchor_count = anchor_count_lf(anchor_count_p[mask][slider_mask], target_anchor_counts)
                    #Masked Coordinate Loss
                    pred_anchors = anchor_coords_p[mask][slider_mask]
                    targ_anchors = y[mask][slider_mask, 11:17]
                    # Create a boolean mask of shape [N, 6] denoting which coordinates are valid
                    valid_coords_mask = torch.zeros_like(pred_anchors, dtype=torch.bool)

                    # Create a boolean mask of shape [N, 6] denoting which coordinates are valid
                    valid_coords_mask = torch.zeros_like(pred_anchors, dtype=torch.bool)
                    for i, count in enumerate(target_anchor_counts):
                        if count > 0:
                            valid_coords_mask[i, :count*2] = True

                    raw_coord_loss = anchor_coord_lf(pred_anchors, targ_anchors)
                    # Average the loss only over the valid anchor coordinates
                    loss_anchor_coords = raw_coord_loss[valid_coords_mask].mean() if valid_coords_mask.any() else 0.0
                else:
                    loss_curve = 0.0
                    loss_anchor_count = 0.0
                    loss_anchor_coords = 0.0
            else:
                loss_obj_type = 0.0
                loss_coords = 0.0
                loss_attr = 0.0
                loss_curve = 0.0

            masked_loss_sum = loss_coords + loss_obj_type + loss_attr + loss_curve + loss_anchor_coords + loss_anchor_count
            total_loss = loss_obj_all + (1 * masked_loss_sum)
            if debug: print(f"Loss obj: {loss_obj_all}, loss_coords: {loss_coords}, loss_obj_type: {loss_obj_type}")
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            epoch_loss += total_loss.item() * len(X)

        avg_train_loss = epoch_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)


        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y, difficulty in test_loader:
                X, y, difficulty = X.to(device), y.to(device), difficulty.to(device)
                is_object = y[:, :, 0].unsqueeze(-1)
                coords = y[:, :, 1:3]
                obj_type = y[:, :, 3:6]   # 3 classes: circle, slider, spinner
                obj_attr = y[:, :, 6:9]
                curve_targets = y[:, :, 9].long()

                is_obj_p, coords_p, type_p, attr_p, curve_p, anchor_count_p, anchor_coords_p = model(X, difficulty)
                loss_obj_all = is_object_lf(is_obj_p, is_object )

                mask = (is_object == 1).squeeze(-1)
                if mask.any():
                    pi_logits, mu, log_sigma = coords_p
                    loss_coords = mdn_loss(pi_logits[mask], mu[mask], log_sigma[mask], coords[mask])

                    valid_type_preds = type_p[mask]
                    valid_type_targets = obj_type[mask]
                    loss_obj_type = obj_type_lf(valid_type_preds, valid_type_targets.argmax(dim=-1))
                    loss_attr = attr_lf(attr_p[mask], obj_attr[mask])

                    slider_mask = (obj_type[mask][:, 1] == 1.0)

                    if slider_mask.any():
                        loss_curve = curve_lf(curve_p[mask][slider_mask], curve_targets[mask][slider_mask])
                        #Anchor Count Loss
                        target_anchor_counts = y[mask][slider_mask, 10].long()
                        loss_anchor_count = anchor_count_lf(anchor_count_p[mask][slider_mask], target_anchor_counts)
                        #Masked Coordinate Loss
                        pred_anchors = anchor_coords_p[mask][slider_mask]
                        targ_anchors = y[mask][slider_mask, 11:17]
                        # Create a boolean mask of shape [N, 6] denoting which coordinates are valid
                        valid_coords_mask = torch.zeros_like(pred_anchors, dtype=torch.bool)

                        # Create a boolean mask of shape [N, 6] denoting which coordinates are valid
                        valid_coords_mask = torch.zeros_like(pred_anchors, dtype=torch.bool)
                        for i, count in enumerate(target_anchor_counts):
                            if count > 0:
                                valid_coords_mask[i, :count*2] = True

                        raw_coord_loss = anchor_coord_lf(pred_anchors, targ_anchors)
                        # Average the loss only over the valid anchor coordinates
                        loss_anchor_coords = raw_coord_loss[valid_coords_mask].mean() if valid_coords_mask.any() else 0.0
                    else:
                        loss_curve = 0.0
                        loss_anchor_count = 0.0
                        loss_anchor_coords = 0.0
                else:
                    loss_obj_type = 0.0
                    loss_coords = 0.0
                    loss_attr = 0.0
                    loss_curve = 0.0
                masked_loss_sum = loss_coords + loss_obj_type + loss_attr + loss_curve + loss_anchor_coords + loss_anchor_count
                total_loss = loss_obj_all + (1 * masked_loss_sum)
                val_loss += total_loss.item() * len(X)

        avg_val_loss = val_loss / len(test_loader.dataset)
        val_losses.append(avg_val_loss)
        print(f"Epoch {epoch:>3} | train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f}")
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            epochs_wo_improvement=0
            torch.save(model.state_dict(), "weights.pth")
        else:
            #epochs_wo_improvement += 1
            pass

    else:
        print(f"Epoch {epoch:>3} | train loss: {avg_train_loss:.4f}")

    torch.cuda.empty_cache()

    return train_losses, val_losses


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#all_samples = glob.glob(r"./processed/**/*.json", recursive=True)
#cust_dataset = CustomDataset(all_samples, hop_length=1024)
train_dataset, test_dataset = CustomDataset.split(
    n_samples=80,
    train_ratio=0.8,
    seed=42,
    window_size=1024,
    debug=False
)


train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=8, shuffle=False)

model = SoundSequenceAnalyzer(time_window_size=1024, freq_vec_len=256, num_heads=32)

train(model, train_loader=train_dataloader, test_loader=test_dataloader, device=device, epochs=100, lr=0.001, debug=False)


In [ ]:
sample_track, _ = CustomDataset.split(
    n_samples=1,
    train_ratio=1,
    seed=42,
    window_size=1024,
    debug=True,
    inference=False
)
sample_loader = DataLoader(sample_track, batch_size=2, shuffle=False)
for X, y, difficulty in sample_loader:
    break

#model = SoundSequenceAnalyzer(time_window_size=1024, freq_vec_len=100, num_heads=10)
#model(X, difficulty)

In [9]:
torch.cuda.empty_cache()


In [ ]:
model = SoundSequenceAnalyzer(time_window_size=4096, freq_vec_len=100, num_heads=10)
state_dict = torch.load('weights.pth', weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()

In [10]:

model.eval()
model.to(device)
    
tp = 0
fp = 0
fn = 0    
with torch.no_grad():
    for X, y, difficulty in dataloader:
        X = X.to(device)
        difficulty = difficulty.to(device)
        
        outputs = model(X, difficulty)
        is_object_pred = outputs[0].to('cpu') # [B, Window_Size, 1]
        
        pred_logits = is_object_pred.squeeze(-1)
        actual_targets = y[:, :, 0] # Ground truth element activity flag
        
        predicted_positive = (pred_logits > threshold)
        actual_positive = (actual_targets == 1.0)
      
        tp += (predicted_positive & actual_positive).sum().item()
        fp += (predicted_positive & ~actual_positive).sum().item()
        fn += (~predicted_positive & actual_positive).sum().item()

# Calculate metrics with zero-division protection fallbacks
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print(f"Precision: {precision}, Recall: {recall}, f1: {f1_score}")
print(f"TP: {tp} FP:{fp} FN: {fn}")

pred: tensor(7485, device='cuda:0')
act: tensor(11270.)


In [24]:
tmp[0].shape

torch.Size([32, 512, 1])

In [11]:


def mdn_sample(pi_logits, mu, log_sigma, temperature=1.0):
    """
    Stochastically samples coordinates from the MDN parameters to inject variety.
    temperature: >1.0 increases randomness, <1.0 makes it closer to standard flow.
    """
    # 1. Sample a mixture component index based on pi probabilities
    pi_dist = torch.distributions.Categorical(logits=pi_logits / temperature)
    component_idx = pi_dist.sample() # Shape: [N]

    # 2. Gather the corresponding mu and log_sigma parameters
    N, K, D = mu.shape  # Batch/Masked elements, Components, Dimensions (2)
    idx = component_idx.unsqueeze(-1).unsqueeze(-1).expand(N, 1, D)

    chosen_mu = mu.gather(-2, idx).squeeze(-2)
    chosen_log_sigma = log_sigma.gather(-2, idx).squeeze(-2)
    chosen_sigma = torch.exp(chosen_log_sigma)

    # 3. Sample from the selected normal distribution
    epsilon = torch.randn_like(chosen_mu)
    sampled_coords = chosen_mu + (epsilon * chosen_sigma * temperature)

    return sampled_coords

In [ ]:
sample_rate = 22050
hop_length = 1024
model = model.to('cpu')
out = model(X, difficulty)
is_object_pred, mdn_out, obj_type_pred, obj_attr_pred = out
pi_logits, mu, log_sigma = mdn_out

mask = (is_object_pred > 0).squeeze(-1)

if mask.any():
    coords_out = mdn_sample(pi_logits[mask], mu[mask])
    pred_classes = torch.argmax(obj_type_pred, dim=-1)
    masked_classes = pred_classes[mask]

    masked_attr = obj_attr_pred[mask]

    mask_idxs = torch.where(mask == True)
    ms = (mask_idxs[1].numpy() * (1000 * hop_length) / sample_rate).astype(int)


In [ ]:
mask[0].sum()

In [ ]:
obj_type = torch.argmax(obj_type_pred, dim=-1)
obj_type_masked = obj_type[mask == True]

In [ ]:
mask_idxs = torch.where(mask==True)
coords_full_shape = torch.zeros(difficulty.shape[0], difficulty.shape[1], 2)
coords_full_shape[mask_idxs[0], mask_idxs[1], :] = coords_out


In [ ]:
is_hit_obj = mask[0] #get where there is a hit object. Get frame index
hit_coords = coords_full_shape[0].squeeze(0)[is_hit_obj]

In [ ]:
hit_obj_idxs = np.where(is_hit_obj == True)[0]

In [ ]:
for idx in range(len(all_samples)):
    if '2138603' in all_samples[idx]:
        print(idx)

In [ ]:
all_samples = glob.glob(r"./processed/**/*.json", recursive=True)


In [ ]:
all_samples = glob.glob(r"./processed/**/*.json", recursive=True)
with open(all_samples[0], 'r') as file:
    data = json.load(file)
data["hit_objects"] = generated_hit_objects


In [12]:
def process_long_sequence(model, X, difficulty, window_size, device="cpu"):
    """
    Slices an audio track sequentially into window_size segments,
    evaluates them via the model, and joins the final predictions.
    """
    model.eval()
    model.to(device)

    # Ensure standard batch dimensions exist [B, Features, Time]
    if X.dim() == 2:
        X = X.unsqueeze(0)
    if difficulty.dim() == 2:
        difficulty = difficulty.unsqueeze(0)

    total_frames = X.shape[-1]
    num_chunks = total_frames // window_size

    # Storage arrays for collecting outputs across loops
    all_is_object = []
    all_pi_logits = []
    all_mu = []
    all_log_sigma = []
    all_obj_types = []
    all_obj_attrs = []
    all_curve_types = []
    all_anchor_count = []
    all_anchor_coords = []
    with torch.no_grad():
        for i in range(num_chunks):
            start = i * window_size
            end = start + window_size

            # Slice step
            X_chunk = X[..., start:end].to(device)
            diff_chunk = difficulty[:, start:end, :].to(device)

            # Evaluate forward pass
            outputs = model(X_chunk, diff_chunk)

            # Accommodates both 3-output and 4-output model versions cleanly

            is_obj_pred, mdn_out, obj_type_pred, obj_attr_pred, curve_type, anchor_count, anchor_coords = outputs
            all_obj_attrs.append(obj_attr_pred.cpu())


            pi_logits, mu, log_sigma = mdn_out

            # Keep on CPU to preserve VRAM during loop steps
            all_is_object.append(is_obj_pred.cpu())
            all_pi_logits.append(pi_logits.cpu())
            all_mu.append(mu.cpu())
            all_log_sigma.append(log_sigma.cpu())
            all_obj_types.append(obj_type_pred.cpu())
            all_curve_types.append(curve_type.cpu())
            all_anchor_coords.append(anchor_coords.cpu())
            all_anchor_count.append(anchor_count.cpu())

    # Concatenate chunk blocks back into standard continuous timelines (dim 1)
    full_is_object = torch.cat(all_is_object, dim=1)
    full_pi_logits = torch.cat(all_pi_logits, dim=1)
    full_mu = torch.cat(all_mu, dim=1)
    full_log_sigma = torch.cat(all_log_sigma, dim=1)
    full_obj_types = torch.cat(all_obj_types, dim=1)
    full_curve_types = torch.cat(all_curve_types, dim=1)
    full_anchor_count = torch.cat(all_anchor_count, dim=1)
    full_anchor_coords = torch.cat(all_anchor_coords, dim=1)
    mdn_recombined = (full_pi_logits, full_mu, full_log_sigma)

    if all_obj_attrs:
        full_obj_attrs = torch.cat(all_obj_attrs, dim=1)
        return (full_is_object, mdn_recombined, full_obj_types, full_obj_attrs, full_curve_types, full_anchor_count, full_anchor_coords)

    return full_is_object, mdn_recombined, full_obj_types

sample_track, _ = CustomDataset.split(
    n_samples=1,
    train_ratio=1,
    seed=42,
    window_size=8096,
    debug=True,
    inference=True
)
sample_loader = DataLoader(sample_track, batch_size=1, shuffle=False)
for X, y, difficulty in sample_loader:
    break
model_out = process_long_sequence(model, X, difficulty, window_size=1024)
is_object_p = model_out[0]
pi_logits, mu, log_sigma = model_out[1]
obj_type_p = model_out[2]
obj_attr_p = model_out[3]
curve_type_p = model_out[4]
anchor_count_p = model_out[5]
anchor_coords_p = model_out[6]

Using debug config
101861/269634


In [13]:
sample_rate = 22050
hop_length = 1024

mask = (is_object_p > 0).squeeze(-1)

if mask.any():
    coords_out = mdn_sample(pi_logits[mask], mu[mask], log_sigma[mask], temperature=0.85)

    # Extract predicted structural types (0=Circle, 1=Slider, 2=Spinner)
    pred_classes = torch.argmax(obj_type_p[mask], dim=-1)

    # Extract geometry mappings across frames
    masked_attributes = obj_attr_p[mask]
    pred_curves = torch.argmax(curve_type_p[mask], dim=-1)
    pred_anchor_counts = torch.argmax(anchor_count_p[mask], dim=-1)
    masked_anchors = anchor_coords_p[mask]

    # Calculate global timestamps in milliseconds from timeline frames
    mask_idxs = torch.where(mask == True)
    ms = (mask_idxs[1].numpy() * (1000 * hop_length) / sample_rate).astype(int)


In [16]:
# --- Complete Upgraded Inference Loop ---

model = model.to('cpu')

if mask.any():
    # FIXED: Apply Stochastic Sampling to completely eliminate Center Position Collapse
    coords_out = mdn_sample(pi_logits[mask], mu[mask], log_sigma[mask], temperature=0.85)

    # Extract predicted structural types (0=Circle, 1=Slider, 2=Spinner)
    pred_classes = torch.argmax(obj_type_p[mask], dim=-1)

    # Extract geometry mappings across frames
    masked_attributes = obj_attr_p[mask]
    pred_curves = torch.argmax(curve_type_p[mask], dim=-1)
    pred_anchor_counts = torch.argmax(anchor_count_p[mask], dim=-1)
    masked_anchors = anchor_coords_p[mask]

    # Calculate global timestamps in milliseconds from timeline frames
    mask_idxs = torch.where(mask == True)
    sample_rate = 22050
    hop_length = 1024
    ms = (mask_idxs[1].numpy() * (1000 * hop_length) / sample_rate).astype(int)

    # Step 2: Build the structural object arrays
    generated_hit_objects = []

    for idx, time_ms in enumerate(ms):
        # Denormalize stochastic starting points
        x = max(0, min(512, int(512 * coords_out[idx, 0].item())))
        y = max(0, min(384, int(384 * coords_out[idx, 1].item())))

        model_class = pred_classes[idx].item()

        obj = {
            "x": x,
            "y": y,
            "time": int(time_ms)
        }

        # Map class integers back to official osu! bit flags
        if model_class == 0:
            obj["type"] = 1 # Circle Flag

        elif model_class == 1:
            obj["type"] = 2 # Slider Flag

            # Extract basic length parameters
            obj["length"] = max(10.0, float(masked_attributes[idx, 2].item() * 1000.0))
            obj["curve_class_idx"] = int(pred_curves[idx].item())

            # Extract anchor counts and their respective coordinates
            num_anchors = int(pred_anchor_counts[idx].item())
            obj["anchor_coords"] = []

            for j in range(num_anchors):
                ax = max(0, min(512, int(512 * masked_anchors[idx, j*2].item())))
                ay = max(0, min(384, int(384 * masked_anchors[idx, (j*2)+1].item())))
                obj["anchor_coords"].append((ax, ay))

            # Keep a fallback traditional endpoint just in case num_anchors calculates to 0
            obj["end_x"] = max(0, min(512, int(512 * masked_attributes[idx, 0].item())))
            obj["end_y"] = max(0, min(384, int(384 * masked_attributes[idx, 1].item())))

        elif model_class == 2:
            obj["type"] = 8 # Spinner Flag
            duration_ms = max(100, int(masked_attributes[idx, 2].item() * 1000.0))
            obj["endTime"] = int(time_ms + duration_ms)

        generated_hit_objects.append(obj)

    # Inject objects back into the target format dict
    all_samples = glob.glob(r"./processed/**/*.json", recursive=True)
    with open(all_samples[1], 'r') as file:
        data = json.load(file)

    data["hit_objects"] = generated_hit_objects

    # Save file
    audio_path = "./sample_map/1022180/audio.mp3"
    json_to_osu(data, "tmp.osu", audio_path)
else:
    print("Zero objects matched structural selection criteria thresholds.")

C:\Users\hiro\Documents\Python projects\vysvetlitelna UI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully generated map with 272 structural elements!


In [15]:
import json
import os

def json_to_osu(parsed_data, output_path, audio_path):
    """
    Converts a parsed dictionary/JSON back into a valid .osu file format.

    :param parsed_data: Dict containing metadata, general, difficulty, and hit_objects.
    :param output_path: Path where the output .osu file will be saved.
    """
    lines = []

    # 1. Write the mandatory file format header (defaulting to v14)
    lines.append("osu file format v14")
    lines.append("")  # Blank line after header

    # 2. Reconstruct Key-Value Sections
    # Order matters slightly for readability, though the game is flexible
    kv_sections = ["general", "metadata", "difficulty"]
    for section_key in kv_sections:
        if section_key in parsed_data and parsed_data[section_key]:
            # Capitalize section name to match standard .osu style (e.g., [General])
            lines.append(f"[{section_key.capitalize()}]")
            for key, value in parsed_data[section_key].items():
                lines.append(f"{key}: {value}")
            lines.append("")  # Blank line after section

    # Generate Timing Point via Audio
    y_audio, sr = librosa.load(audio_path)
    tempo, _ = librosa.beat.beat_track(y=y_audio, sr=sr)
    bpm = float(tempo[0]) if isinstance(tempo, np.ndarray) else float(tempo)
    beat_length = 60000.0 / bpm if bpm > 0 else 500.0

    lines.append("[TimingPoints]")
    # Format: time, beatLength, meter, sampleSet, sampleIndex, volume, uninherited, effects
    lines.append(f"0,{beat_length},4,2,1,60,1,0")
    lines.append("")

    # Reconstruct HitObjects
    if "hit_objects" in parsed_data and parsed_data["hit_objects"]:
        lines.append("[HitObjects]")
        for obj in parsed_data["hit_objects"]:
            x = max(0, min(512, int(obj.get("x", 256))))
            y = max(0, min(384, int(obj.get("y", 192))))
            time = int(obj.get("time", 0))
            obj_type = int(obj.get("type", 1)) # Extracted from the model's argmax(obj_type)

            # Base parameters
            parts = [str(x), str(y), str(time), str(obj_type), "0"]

            inv_curve_map = {0: 'L', 1: 'B', 2: 'P', 3: 'C'}

            if obj_type == 2: # Slider (Assuming argmax gave us 2 for bit flag 2)
                length = max(10.0, float(obj.get("length", 100.0)))
                curve_letter = inv_curve_map.get(int(obj.get("curve_class_idx", 0)), 'L')

                # Fetch structural coordinates array
                anchors = obj.get("anchor_coords", [])

                if len(anchors) > 0:
                    # Construct multi-node anchor string format (e.g., B|200:100|250:300)
                    anchor_strings = [f"{ax}:{ay}" for ax, ay in anchors]
                    curve_syntax = f"{curve_letter}|" + "|".join(anchor_strings)
                else:
                    # Traditional fallback endpoint if zero anchors were predicted
                    end_x = max(0, min(512, int(obj.get("end_x", x))))
                    end_y = max(0, min(384, int(obj.get("end_y", y))))
                    curve_syntax = f"{curve_letter}|{end_x}:{end_y}"

                parts.append(curve_syntax)
                parts.append("1") # Default standard repeat slider loop count
                parts.append(str(length))

            elif obj_type == 8: # Spinner
                end_time = time + max(100, int(obj.get("attr_val", 1000)))
                parts.append(str(end_time))

            lines.append(",".join(parts))
        lines.append("")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Successfully generated map with {len(generated_hit_objects)} structural elements!")


In [ ]:
tmp = 1

In [ ]:
print(tmp)